In [1]:
using TextAnalysis: serialize
using SQLite
using DataFrames
using WordTokenizers
using StatsBase
using Serialization
using MLDataUtils
using TextAnalysis
using ProgressMeter
using MLJ
using MLJText
using MLJBase
using Languages
using ThreadsX

CountTransformer = @load CountTransformer pkg=MLJText
MultinomialNBClassifier = @load MultinomialNBClassifier pkg=NaiveBayes

[ Info: For silent loading, specify `verbosity=0`. 


import MLJText ✔
import MLJNaiveBayesInterface ✔

[ Info: For silent loading, specify `verbosity=0`. 


MLJNaiveBayesInterface.MultinomialNBClassifier

In [2]:
function load_data()
    println("Fetching data from db...")
    db = SQLite.DB("data/panslop.db")
    spam_db = DBInterface.execute(db, "SELECT * FROM full_text ORDER BY RANDOM() LIMIT 2000") |> DataFrame
    ham_db = DBInterface.execute(db, "SELECT * FROM ham_full_text ORDER BY RANDOM() LIMIT 2000") |> DataFrame

    # ham_linux_docs = read("ham/LINUX_DOCS.md", String)
    # println("Loaded.")

    spam = spam_db.text
    ham = ham_db.text
    # push!(ham, ham_linux_docs)

    return spam, ham
end

load_data (generic function with 1 method)

In [3]:
spam, ham = load_data()
println("$(length(spam)) spam files")
println("$(length(ham)) ham files")

Fetching data from db...
2000 spam files
2000 ham files


In [4]:
# split test and train set with Julia's cool new MLDataUtils
# refs:
# https://discourse.julialang.org/t/simple-tool-for-train-test-split/473/4
# https://github.com/JuliaML/MLDataUtils.jl
train_ham, test_ham = splitobs(ham; at=0.8)
train_spam, test_spam = splitobs(spam; at=0.8)

(["# Changelog\n\nAll notable changes to this project will be documented in this file.\n\nThe format is based on [Keep a Changelog](https://keepachangelog.com/en/1.1.0/),\nand this project adheres to [Semantic Versioning](https://semver.org/spec/v2.0.0.html).\n\n## [Unreleased]\n\n### Added\n- More news frontpage presets for briefing browse sources: the pick-list now offers 18 reputable sources (added BBC, NPR, Deutsche Welle, France 24, The Japan Times, South China Morning Post, CNBC, Politico, Axios, Techmeme, and Hacker News), grouped by region and beat. You can still point a browse source at any URL.\n- Shared curated content for briefings. Content that's identical for everyone — world headlines, a markets snapshot, a curated digest — can now be generated once and read by every user's briefing, instead of each briefing fetching and summarizing it separately. Add a **Shared block** source to any briefing block and pick from the available shared content. Admins manage the shared bloc

In [5]:
# prepare labels on the train set
train_labels = vcat(repeat(["ham"], length(train_ham)), repeat(["spam"], length(train_spam)))
test_labels = vcat(repeat(["ham"], length(test_ham)), repeat(["spam"], length(test_spam)))

800-element Vector{String}:
 "ham"
 "ham"
 "ham"
 "ham"
 "ham"
 "ham"
 "ham"
 "ham"
 "ham"
 "ham"
 "ham"
 "ham"
 "ham"
 ⋮
 "spam"
 "spam"
 "spam"
 "spam"
 "spam"
 "spam"
 "spam"
 "spam"
 "spam"
 "spam"
 "spam"
 "spam"

In [6]:
corpus_train = vcat(train_spam, train_ham)
corpus_test = vcat(test_spam, test_ham)

800-element Vector{String}:
 "# DuDuClaw 🐾\n\n<div align=\"cent" ⋯ 15340 bytes ⋯ "  🐾 Built with louis.li\n</p>\n"
 "# Contributing to Blocks Beyon" ⋯ 6841 bytes ⋯ "d ask — we are happy to help.\n"
 "# OneSub Repository Guide\n\nThi" ⋯ 29560 bytes ⋯ "rkflow\nhas no `npm ci` step.\n"
 "# Support\n\nJustSearch is **alp" ⋯ 1688 bytes ⋯ "on may\n  already be answered.\n"
 "## Security Announcements\n\nJoi" ⋯ 2254 bytes ⋯ "en setting a disclosure date.\n"
 "# Agents and Execution Archite" ⋯ 9383 bytes ⋯ "EBUGGING.md](DEBUGGING.md)**.\n"
 "# Joanium — Claude Code Instruc" ⋯ 609 bytes ⋯ "ad `Docs/` for full internals\n"
 "# Lottie Studio\n\n**Create anim" ⋯ 7582 bytes ⋯ "ain`.\n\n---\n\n## 📝 License\n\nMIT\n"
 "# Full-System Audit Report\n\nDa" ⋯ 15934 bytes ⋯ "the_fast_providers_response`\n"
 "# Play South Wales - League Sc" ⋯ 2029 bytes ⋯ "ker compose up --build\n   ```\n"
 "# Bento — self-contained offic" ⋯ 42102 bytes ⋯ "e starting a synthetic drag.\n"
 "# Open exchange-rate sources\n\n" 

In [7]:
println("Tokenising...")
tokenised_train = ThreadsX.map(doc -> TextAnalysis.tokenize(Languages.English(), doc), corpus_train)
tokenised_test = ThreadsX.map(doc -> TextAnalysis.tokenize(Languages.English(), doc), corpus_test)

Tokenising...


800-element Vector{Vector{String}}:
 ["#", "DuDuClaw", "🐾", "<div", "align=", "\"", "center", "\"", ">", "["  …  "align=", "\"", "center", "\"", ">", "🐾", "Built", "with", "louis.li", "</p>"]
 ["#", "Contributing", "to", "Blocks", "Beyond", "the", "Stars", "Thanks", "for", "your"  …  "issue", "and", "ask", "—", "we", "are", "happy", "to", "help", "."]
 ["#", "OneSub", "Repository", "Guide", "This", "is", "the", "canonical", "repository", "guide"  …  "that", "workflow", "has", "no", "`", "npm", "ci", "`", "step", "."]
 ["#", "Support", "JustSearch", "is", "*", "*", "alpha", "*", "*", "software"  …  "https://github.com/eliasjustus/justsearch/issues", ")", "—", "your", "question", "may", "already", "be", "answered", "."]
 ["#", "#", "Security", "Announcements", "Join", "the", "[", "llm-d-security-announce", "]", "("  …  "hold", "the", "final", "say", "when", "setting", "a", "disclosure", "date", "."]
 ["#", "Agents", "and", "Execution", "Architecture", ">", "*", "*", "⚠️", "Debugging"  … 

In [8]:
println("Computing features...")
mach1 = machine(CountTransformer(), tokenised_train) |> MLJ.fit!

# matrix of counts
X = MLJ.transform(mach1, tokenised_train)
y = coerce(train_labels, OrderedFactor)

Computing features...


[ Info: Training machine(CountTransformer(max_doc_freq = 1.0, …), …).


3200-element CategoricalArrays.CategoricalArray{String,1,UInt32}:
 "ham"
 "ham"
 "ham"
 "ham"
 "ham"
 "ham"
 "ham"
 "ham"
 "ham"
 "ham"
 "ham"
 "ham"
 "ham"
 ⋮
 "spam"
 "spam"
 "spam"
 "spam"
 "spam"
 "spam"
 "spam"
 "spam"
 "spam"
 "spam"
 "spam"
 "spam"

In [9]:
classifier = MultinomialNBClassifier()

MultinomialNBClassifier(
  alpha = 1)

In [10]:
mach2 = machine(classifier, X, y)

untrained Machine; caches model-specific representations of data
  model: MultinomialNBClassifier(alpha = 1)
  args: 
    1:	Source @936 ⏎ AbstractMatrix{Count}
    2:	Source @383 ⏎ AbstractVector{OrderedFactor{2}}


In [11]:
MLJ.fit!(mach2, rows=1:length(corpus_train))

[ Info: Training machine(MultinomialNBClassifier(alpha = 1), …).


trained Machine; caches model-specific representations of data
  model: MultinomialNBClassifier(alpha = 1)
  args: 
    1:	Source @936 ⏎ AbstractMatrix{Count}
    2:	Source @383 ⏎ AbstractVector{OrderedFactor{2}}


In [12]:
serialize("data/model.dat", mach2)